# 30 Merge And Validate Full Dataset

Merges identity + open corpora, computes row/token composition, enforces targets,
and writes final artifacts.

Required outputs:
- `workspace/final/full_sft.jsonl`
- `workspace/final/full_preferences.jsonl`
- `workspace/final/dataset_manifest.json`
- `workspace/reports/full_dataset_validation.json`
- `workspace/reports/mixture_math.json`


In [ ]:

from __future__ import annotations

import json
from pathlib import Path

import yaml

from lumis1.full_dataset import (
    build_dataset_manifest,
    build_full_dataset_validation_report,
    infer_modality,
    select_open_sft_rows,
)
from lumis1.identity_pack import iter_jsonl, normalize_identity_preference_row, resolve_identity_paths

REPO_ROOT = Path.cwd().resolve()
REPORTS = REPO_ROOT / "workspace" / "reports"
FINAL = REPO_ROOT / "workspace" / "final"
INTERIM = REPO_ROOT / "workspace" / "interim"
REPORTS.mkdir(parents=True, exist_ok=True)
FINAL.mkdir(parents=True, exist_ok=True)

ALLOW_SMALL_SAMPLE = False

mixture = yaml.safe_load((REPO_ROOT / "configs" / "dataset_mixture.yaml").read_text(encoding="utf-8"))
identity_report = json.loads((REPORTS / "identity_validation.json").read_text(encoding="utf-8"))
identity_paths = resolve_identity_paths(REPO_ROOT)
identity_sft_path = Path(identity_paths["sft"])
identity_pref_path = Path(identity_paths["preferences"])
open_sft_path = INTERIM / "open_sft.jsonl"
open_pref_path = INTERIM / "open_preferences.jsonl"

for path in (identity_sft_path, identity_pref_path, open_sft_path, open_pref_path):
    if not path.exists():
        raise FileNotFoundError(f"missing required input for merge: {path}")

identity_sft = []
for row in iter_jsonl(identity_sft_path):
    item = dict(row)
    item.setdefault("category", "identity_behavior")
    item["modality"] = infer_modality(item)
    identity_sft.append(item)

open_sft_all = []
for row in iter_jsonl(open_sft_path):
    item = dict(row)
    item["modality"] = infer_modality(item)
    open_sft_all.append(item)

identity_pref = [
    normalize_identity_preference_row(row, idx)
    for idx, row in enumerate(iter_jsonl(identity_pref_path), start=1)
]
open_pref = list(iter_jsonl(open_pref_path))
full_pref = identity_pref + open_pref

selection = select_open_sft_rows(
    identity_rows=identity_sft,
    open_rows=open_sft_all,
    identity_share_target=float(mixture["identity_pack"].get("fixed_share_of_final_sft_tokens", 0.20)),
    allow_small_sample=ALLOW_SMALL_SAMPLE,
)
selected_open = selection["selected_open_rows"]
full_sft = identity_sft + selected_open

full_sft_path = FINAL / "full_sft.jsonl"
full_pref_path = FINAL / "full_preferences.jsonl"
with full_sft_path.open("w", encoding="utf-8") as handle:
    for row in full_sft:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
with full_pref_path.open("w", encoding="utf-8") as handle:
    for row in full_pref:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

validation_report = build_full_dataset_validation_report(
    REPO_ROOT,
    full_sft_path=full_sft_path,
    full_preferences_path=full_pref_path,
    identity_validation_report=identity_report,
    allow_small_sample=ALLOW_SMALL_SAMPLE,
)
if not validation_report["pass"]:
    raise RuntimeError(
        "full dataset validation failed:\n" + json.dumps(validation_report["validations"], indent=2)
    )

validation_report_path = REPORTS / "full_dataset_validation.json"
validation_report_path.write_text(json.dumps(validation_report, indent=2), encoding="utf-8")
mixture_math = {
    "counts": validation_report["counts"],
    "shares": validation_report["shares"],
    "targets": validation_report["targets"],
    "selection": {
        "mode": selection["selection_mode"],
        "required_open_tokens_exact": selection["required_open_tokens_exact"],
        "open_tokens_selected": selection["open_tokens"],
        "remaining_open_tokens": selection["remaining_open_tokens"],
    },
}
(REPORTS / "mixture_math.json").write_text(json.dumps(mixture_math, indent=2), encoding="utf-8")

manifest = build_dataset_manifest(
    full_sft_path=full_sft_path,
    full_preferences_path=full_pref_path,
    validation_report=validation_report,
    validation_report_path=validation_report_path,
)
(FINAL / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps({"selection": mixture_math["selection"], "validation": validation_report}, indent=2))
print("saved:", FINAL / "dataset_manifest.json")
